In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import optuna

from lightgbm import LGBMRegressor
from scipy.stats import pearsonr

optuna.logging.set_verbosity(optuna.logging.WARNING)

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]

N_TRIALS = 50

In [ ]:
def train(X_train, y_train, model_directory_path, loop_moon, embargo):

    feature_cols = [
        c for c in X_train.columns
        if c.startswith("Feature_")
    ]

    df = X_train.merge(
        y_train[["moon", "id", "target"]],
        on=["moon", "id"]
    )

    df = df.dropna(subset=["target"])

    all_moons = sorted(df["moon"].unique())

    cutoff = int(len(all_moons) * 0.80)

    tr_moons  = all_moons[:cutoff]
    val_moons = all_moons[cutoff:]

    df_tr  = df[df["moon"].isin(tr_moons)]
    df_val = df[df["moon"].isin(val_moons)]

    X_tr = df_tr[feature_cols].fillna(0)
    y_tr = df_tr["target"]

    X_val = df_val[feature_cols].fillna(0)

    val_moon_arr   = df_val["moon"].values
    val_target_arr = df_val["target"].values

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True
            ),

            "num_leaves": trial.suggest_int(
                "num_leaves",
                15,
                255
            ),

            "min_child_samples": trial.suggest_int(
                "min_child_samples",
                20,
                200
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.5,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.4,
                1.0
            ),

            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                1e-4,
                10.0,
                log=True
            ),

            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                1e-4,
                10.0,
                log=True
            ),
        }

        model = LGBMRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        corrs = []

        for moon in val_moons:

            mask = val_moon_arr == moon

            if mask.sum() < 2:
                continue

            r, _ = pearsonr(
                preds[mask],
                val_target_arr[mask]
            )

            if not np.isnan(r):
                corrs.append(r)

        return float(np.mean(corrs)) if corrs else -1.0

    print(
        f"  Optuna: {N_TRIALS} trials  "
        f"(train={len(tr_moons)} moons, "
        f"val={len(val_moons)} moons)"
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=False
    )

    best_params = study.best_params
    best_val_ic = study.best_value

    print(f"  Best val IC  = {best_val_ic:+.5f}")
    print(f"  Best params  = {best_params}")

    model = LGBMRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(
        df[feature_cols].fillna(0),
        df["target"]
    )

    joblib.dump(
        {
            "model": model,
            "features": feature_cols,
            "best_params": best_params,
            "best_val_ic": best_val_ic,
        },
        os.path.join(model_directory_path, "model.joblib")
    )

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj = joblib.load(
        os.path.join(model_directory_path, "model.joblib")
    )

    model    = obj["model"]
    features = obj["features"]

    preds = model.predict(
        X_test[features].fillna(0)
    )

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })

In [ ]:
import os
import numpy as np
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"

os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

obj = joblib.load(
    os.path.join(model_dir, "model.joblib")
)

print(f"\nBest params : {obj['best_params']}")
print(f"Best val IC : {obj['best_val_ic']:.4f}\n")

results = []

for moon in splits["reduced_cloud"]:

    pred = infer(
        X_test_cloud[X_test_cloud["moon"] == moon],
        model_dir,
        loop_moon=moon,
        embargo=embargo
    )

    results.append(pred)

predictions = pd.concat(results)

corrs = []

for moon in splits["reduced_cloud"]:

    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon],
        on=["moon", "id"]
    )

    if len(merged) > 1:

        r, _ = pearsonr(
            merged["prediction"],
            merged["target"]
        )

        corrs.append(r)

        print(f"Moon {moon}: Pearson r = {r:.4f}")

print(f"\nMean IC = {np.mean(corrs):.4f}")